In [1]:
import shutil
import os
import sys

In [2]:
shutil.which("dggrid")

'/Users/akmoch/dev/build/igeo7_z7_xarray_paper/.pixi/envs/default/bin/dggrid'

In [3]:
os.environ["DGGRID_PATH"]

'/Users/akmoch/dev/build/igeo7_z7_xarray_paper/.pixi/envs/default/bin/dggrid'

In [4]:
dggrid_exec = os.environ["DGGRID_PATH"]

In [5]:
sys.path.append("/Users/akmoch/dev/build/igeo7_z7_xarray_paper")

In [6]:
sys.path.append("/Users/akmoch/dev/build/igeo7_z7_xarray_paper/src")

In [7]:
from dggrid4py import DGGRIDv7
import numpy as np

In [8]:
dggrid_instance = DGGRIDv7(executable=dggrid_exec, working_dir='.', capture_logs=False, silent=False, tmp_geo_out_legacy=False, debug=False)

In [9]:
df = dggrid_instance.grid_stats_table('ISEA3H', 30)
df["pixil_m"] =  np.sqrt(df["Area (km^2)"] * 1000000)
df

** executing DGGRID version 8.44 with GDAL version 3120300 **
type sizes: big int: 64 bits / big double: 64 bits

** using meta file metafile_128a3f58-1fde-4a70-82fb-14e54e114593...
* parameter values:
dggrid_operation OUTPUT_STATS (user set)
rng_type RAND (default)
precision 7 (user set)
verbosity 0 (default)
pause_on_startup false (default)
pause_before_exit false (default)
update_frequency 100000 (default)
dggs_type ISEA3H (user set)
dggs_topology HEXAGON (user set)
dggs_proj ISEA (user set)
dggs_aperture_type PURE (user set)
dggs_aperture 3 (user set)
proj_datum WGS84_AUTHALIC_SPHERE (default)
dggs_orient_specify_type SPECIFIED (user set)
dggs_num_placements 1 (user set)
dggs_vert0_lon 11.25 (user set)
dggs_vert0_lat 58.2825 (user set)
dggs_vert0_azimuth 0 (user set)
dggs_res_specify_type SPECIFIED (user set)
dggs_res_spec 30 (user set)
z3_invalid_digit 0 (default)

Earth Radius: 6,371.0071809

Res               # Cells        Area (km^2)      CLS (km)
0                    12 51,00

,Resolution,Cells,Area (km^2),CLS (km),pixil_m
0,0,12,5.100656e+07,8199.500370,7.141888e+06
1,1,32,1.700219e+07,4678.969872,4.123371e+06
2,2,92,5.667396e+06,2691.252071,2.380629e+06
3,3,272,1.889132e+06,1551.867549,1.374457e+06
4,4,812,6.297106e+05,895.601842,7.935431e+05
5,5,2432,2.099035e+05,517.004997,4.581523e+05
6,6,7292,6.996785e+04,298.479323,2.645144e+05
7,7,21872,2.332262e+04,172.324491,1.527174e+05
8,8,65612,7.774205e+03,99.491086,8.817146e+04
9,9,196832,2.591402e+03,57.441108,5.090581e+04


In [10]:
from z7py import z7 as Z7
import numpy as np

Z7._INVALID_RAW

np.uint64(18446744073709551615)

In [11]:
import xarray as xr
import numpy as np

# These imports register the xdggs accessor + the IGEO7 grid backend
import xdggs                                          # noqa: F401  -> ds.dggs.*
from xdggs_dggrid4py.index import IGEO7Index          # noqa: F401  -> registers 'igeo7'

import z7_xarray_paper.z7_zarr as z7_zarr


In [12]:
import pandas as pd
import numpy as np
import zarr

from z7_xarray_paper.distance_measures import HexGridDistortionModel
from z7_xarray_paper.kernels import slope as z7_slope, slope_blocked as z7_slope_blocked

DIST_PARQUET = "../data/working/dist_lookup_level4.parquet"
model = HexGridDistortionModel(
    pd.read_parquet(DIST_PARQUET).to_dict("index")
)
print(f"Distortion model loaded: {len(model.region_weights)} level-4 parents")

Distortion model loaded: 24012 level-4 parents


In [13]:
ds = z7_zarr.open_dataset("../data/working/pori_z7_r12_ranges.zarr").chunk({"cell_ids": 117649}).unify_chunks()
ds

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


<xarray.Dataset> Size: 2MB
Dimensions:    (cell_ids: 158430)
Coordinates:
  * cell_ids   (cell_ids) uint64 1MB dask.array<chunksize=(117649,), meta=np.ndarray>
Data variables:
    elevation  (cell_ids) float32 634kB dask.array<chunksize=(117649,), meta=np.ndarray>
Indexes:
    cell_ids  Z7MonotonicIndex(level=12, R=1073, N=158430)
Attributes:
    clipper_scale_factor:  100000000
    dggs:                  {'compression': 'ranges', 'coordinate': 'cell_id_r...
    regridder:             xdggs_dggrid4py.mapblocks_nearestcentroid
    source_crs:            EPSG:3301
    source_path:           /Users/akmoch/dev/build/igeo7_z7_xarray_paper/data...
    zarr_conventions:      [{'description': 'Discrete Global Grid Systems con...

In [14]:
ds.dggs.index

In [15]:
slope_da = z7_slope_blocked(ds["elevation"], model, distance_mode="lookup")

In [16]:
slope_da

<xarray.DataArray 'elevation' (cell_ids: 158430)> Size: 1MB
dask.array<concatenate, shape=(158430,), dtype=float64, chunksize=(117649,), chunktype=numpy.ndarray>
Coordinates:
  * cell_ids  (cell_ids) uint64 1MB dask.array<chunksize=(158430,), meta=np.ndarray>
Indexes:
    cell_ids  Z7MonotonicIndex(level=12, R=1073, N=158430)
Attributes:
    units:      m/m
    long_name:  slope magnitude (FDA)

In [17]:
# slope_da.to_zarr("pori_z7_r12_slope.zarr")  # triggers compute chunk-by-chunk

In [18]:
ds["slope"] = slope_da

In [19]:
ds["elevation"] = ds["elevation"].compute()
ds["slope"] = ds["slope"].compute()

In [20]:
ds.dggs.explore()

/Users/akmoch/dev/build/igeo7_z7_xarray_paper/.pixi/envs/default/lib/python3.13/site-packages/dask/array/core.py:1738: FutureWarning: The `numpy.astype` function is not implemented by Dask array. You may want to use the da.map_blocks function or something similar to silence this warning. Your code may stop working in a future release.
  warnings.warn(
/Users/akmoch/dev/build/igeo7_z7_xarray_paper/.pixi/envs/default/lib/python3.13/site-packages/lonboard/_geoarrow/ops/reproject.py:40: UserWarning: No CRS exists on data. If no data is shown on the map, double check that your CRS is WGS84.
  warn(


In [21]:
ds

<xarray.Dataset> Size: 3MB
Dimensions:    (cell_ids: 158430)
Coordinates:
  * cell_ids   (cell_ids) uint64 1MB dask.array<chunksize=(158430,), meta=np.ndarray>
Data variables:
    elevation  (cell_ids) float32 634kB 74.0 74.0 75.8 ... 106.5 108.9 107.0
    slope      (cell_ids) float64 1MB 0.01525 0.01525 0.01694 ... nan nan
Indexes:
    cell_ids  Z7MonotonicIndex(level=12, R=1073, N=158430)
Attributes:
    clipper_scale_factor:  100000000
    dggs:                  {'compression': 'ranges', 'coordinate': 'cell_id_r...
    regridder:             xdggs_dggrid4py.mapblocks_nearestcentroid
    source_crs:            EPSG:3301
    source_path:           /Users/akmoch/dev/build/igeo7_z7_xarray_paper/data...
    zarr_conventions:      [{'description': 'Discrete Global Grid Systems con...